In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#### Import Library

In [ ]:
from pathlib import Path
import gc
import torch
import pandas as pd
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    set_seed,
)

In [ ]:
# !unzip -q "/content/drive/MyDrive/Colab Notebooks/Cleaned_output.zip" -d "/content/drive/MyDrive/Colab Notebooks/Cleaned_data"

#### Configuration

In [ ]:
RND = 42
set_seed(RND)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

BATCH_SIZE = 64

COMMENTS_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Cleaned_data/digikala-comments_parts")
OUTPUT_DIR = "/content/drive/MyDrive/Cleaned_data/Comments_ABSA"

# OUTPUT_DIR.mkdir(
#     parents=True,
#     exist_ok=True
# )

SENTIMENT_MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/pars_absa_sentiment_model"
ASPECT_MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/pars_absa_aspect_model"

GPU: False 


#### Load Data

In [ ]:
parquet_files = sorted(COMMENTS_DIR.glob("*.parquet"))

print(f"Found {len(parquet_files)} parquet files.\n")

for file in parquet_files[:5]:
    print(file.name)

Found 124 parquet files.

part_0000.parquet
part_0001.parquet
part_0002.parquet
part_0003.parquet
part_0004.parquet


In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    SENTIMENT_MODEL_PATH,
    use_fast = True,
)

print("Loading Sentiment Model...")
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    SENTIMENT_MODEL_PATH
).to(device)

print("Loading Aspect Model...")
aspect_model = AutoModelForTokenClassification.from_pretrained(
    SENTIMENT_MODEL_PATH
).to(device)

sentiment_model.eval()
aspect_model.eval()

print("Models Loaded!")

Loading tokenizer...
Loading Sentiment Model...


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Loading Aspect Model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: /content/drive/MyDrive/Colab Notebooks/pars_absa_sentiment_model
Key                        | Status     | 
---------------------------+------------+-
classifier.dense.weight    | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.weight          | MISSING    | 
classifier.bias            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Models Loaded!


In [ ]:
# SENTIMENT_ID2LABEL = {
#     0: "negative",
#     1: "neutral",
#     2: "positive",
# }

# BIO_ID2LABEL = {
#     0: "O",
#     1: "B-ASP",
#     2: "I-ASP",
# }

print(sentiment_model.config.id2label)
print(aspect_model.config.id2label)

{0: 'negative', 1: 'neutral', 2: 'positive'}
{0: 'negative', 1: 'neutral', 2: 'positive'}


In [ ]:
@torch.no_grad()
def predict_sentiment(texts):
  encoded = tokenizer(
      texts,
      padding = True,
      truncation = True,
      max_length = 128,
      return_tensors = "pt",
  ).to(device)
  encoded = {k: v.to(device) for k, v in encoded.items()}
  outputs = sentiment_model(**encoded)
  preds = outputs.logits.argmax(dim=1)
  preds = preds.cpu().numpy()

  labels = [
      SENTIMENT_ID2LABEL[p]
      for p in preds
  ]

  return labels

In [ ]:
sample = [
    "این گوشی فوق العاده است.",
    "باطری خیلی ضعیفه.",
    "قیمتش مناسبه."
]

predict_sentiment(sample)

NameError: name 'SENTIMENT_ID2LABEL' is not defined

In [ ]:
file = parquet_files[0]
df = pd.read_parquet(file)
print(df.shape)
df.head()

In [ ]:
texts = df["raw_text_normalized"].fillna("").tolist()

In [ ]:
def batch_generator(items, batch_size):

    for i in range(0, len(items), batch_size):

        yield items[i:i + batch_size]


for batch in batch_generator(texts, BATCH_SIZE):

    preds = predict_sentiment(batch)

    print(preds[:5])

    break

In [ ]:
@torch.no_grad()
def predict_aspects(texts):

    encoded = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=128,
        return_offsets_mapping=True,
        return_tensors="pt"
    )

    offsets = encoded.pop("offset_mapping")

    encoded = {
        k: v.to(device)
        for k, v in encoded.items()
    }

    outputs = aspect_model(**encoded)

    preds = outputs.logits.argmax(-1).cpu().numpy()

    input_ids = encoded["input_ids"].cpu().numpy()

    return preds, input_ids, offsets

In [ ]:
def decode_aspects(text, pred_ids, offsets):

    aspects = []

    current = ""

    for pred, (start, end) in zip(pred_ids, offsets):

        if end == 0:
            continue

        label = BIO_ID2LABEL[int(pred)]

        token = text[start:end]

        if label == "B-ASP":

            if current:
                aspects.append(current)

            current = token

        elif label == "I-ASP":

            current += token

        else:

            if current:
                aspects.append(current)
                current = ""

    if current:
        aspects.append(current)

    return aspects

In [ ]:
sample = [
    "باتری گوشی فوق العاده است."
]

preds, ids, offsets = predict_aspects(sample)

decode_aspects(
    sample[0],
    preds[0],
    offsets[0]
)

In [ ]:
for file in parquet_files:

    print("=" * 70)
    print(file.name)

    df = pd.read_parquet(file)

    texts = (
        df["raw_text_normalized"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    print(len(texts))

In [ ]:
all_sentiments = []
all_aspects = []

In [ ]:
for batch in batch_generator(texts, BATCH_SIZE):

    sentiments = predict_sentiment(batch)

    preds, ids, offsets = predict_aspects(batch)

    for i, text in enumerate(batch):

        aspects = decode_aspects(
            text,
            preds[i],
            offsets[i]
        )

        all_aspects.append(aspects)
    all_sentiments.extend(sentiments)



In [ ]:
len(all_sentiments)
len(all_aspects)
len(df)

In [ ]:
df["predicted_sentiment"] = all_sentiments
df["predicted_aspects"] = all_aspects

In [ ]:
output_path = Path(OUTPUT_DIR) / file.name

df.to_parquet(
    output_path,
    index=False,
    engine="pyarrow"
)

print(f"Saved -> {output_path}")

In [ ]:
del df
del texts

gc.collect()
torch.cuda.empty_cache()